# Person 2 (Revision 2) — Classical Acoustic + Topic/Context Evidence, on real SLR54 data

This notebook runs Person 2's pipeline against the **actual** data provided:
`audio_index.tsv` (157,905-row SLR54 manifest), `whisper_output.tsv`, and
`whisper_words.tsv`. See `person2_task_v2.md` for the full rewritten task
write-up; the short version:

- **`audio_index.tsv`** supplies the real broad Nepali corpus for LDA — no
  synthetic text is used anywhere in Part B.
- No file provides pre-built confusable-word clusters, so **Step 0** below
  discovers plausible clusters directly from the real vocabulary (minimal
  orthographic pairs), then mines real occurrences for them from
  `audio_index.tsv`.
- No file provides `speaker_id` or frame-level acoustic features. The
  notebook is explicit about both gaps: it uses an occurrence-level split
  instead of a speaker-disjoint one, and Part A's GMM evidence will be real
  (via `librosa`) the moment `.flac` audio is reachable at `AUDIO_ROOT` —
  until then it reports zero available audio and continues without error.

Every cell below is written to run to completion with the current data. If
you later have access to the `.flac` files, set `AUDIO_ROOT` and re-run —
nothing else changes.


0. Setup

In [ ]:
!pip install -q gensim librosa soundfile joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 54.1 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import time
import warnings
import unicodedata
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.special import softmax

from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture

from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

import librosa
import joblib

warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)


## 1. Configuration + load the three real files

Upload `audio_index.tsv`, `whisper_output.tsv`, and `whisper_words.tsv` to
Colab (drag into the file browser, or run the upload cell below), then set
`DATA_DIR` to wherever they land (`/content` by default).


In [ ]:
DATA_DIR = "/content"  #@param {type:"string"}

# Uncomment to use Colab's upload widget instead of drag-and-drop:
# from google.colab import files
# uploaded = files.upload()

NUM_CLUSTERS = 20            #@param {type:"integer"}   # how many discovered clusters to run the pipeline on
MAX_OCC_PER_CANDIDATE = 150  #@param {type:"integer"}   # cap occurrences mined per candidate word
MIN_WORD_FREQ = 5            #@param {type:"integer"}   # a candidate word must appear at least this often
MIN_LEN, MAX_LEN = 2, 10                                  # candidate word length range (Devanagari chars)
LDA_CORPUS_SIZE = 40000      #@param {type:"integer"}   # subsample size of the 157,905 real sentences for LDA
AUDIO_ROOT = "/content/asr_nepali_audio"  #@param {type:"string"}  # set once .flac files are reachable

OUTPUT_DIR = "/content/person2_outputs"
ARTIFACT_DIR = os.path.join(OUTPUT_DIR, "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

audio_index = pd.read_csv(os.path.join(DATA_DIR, "audio_index.tsv"), sep="\t",
                           quoting=3, on_bad_lines="skip")
whisper_output = pd.read_csv(os.path.join(DATA_DIR, "whisper_output.tsv"), sep="\t")
whisper_words = pd.read_csv(os.path.join(DATA_DIR, "whisper_words.tsv"), sep="\t")
whisper_words["word"] = whisper_words["word"].str.strip()

print(f"audio_index: {len(audio_index)} rows | whisper_output: {len(whisper_output)} | "
      f"whisper_words: {len(whisper_words)}")

DEV_RE = re.compile(r"^[\u0900-\u097F]+$")
def is_devanagari(s):
    return bool(DEV_RE.match(str(s)))


audio_index: 157905 rows | whisper_output: 652 | whisper_words: 2646


## Step 0 — Discover confusable word clusters from the real vocabulary

No file supplies a curated confusable-word list, so this step bootstraps one:
group words that are a single-character edit apart (after excluding common
case-marker suffixes, which would otherwise merge inflectional forms of the
same word into giant, useless blobs), keep small clusters (2–4 words) as
plausible genuine confusability, and take the top `NUM_CLUSTERS` by combined
corpus frequency. See `person2_task_v2.md` Section 3 for the full rationale.


In [ ]:
all_tokens = [t for s in audio_index["reference"].astype(str) for t in s.split()]
vocab = Counter(t for t in all_tokens if is_devanagari(t))

SUFFIXES = ["हरूलाई", "हरूको", "हरूमा", "हरूका", "हरूले", "हरू",
            "बाट", "सम्म", "सँग", "लाई", "बाटै", "मध्ये",
            "कै", "मै", "की", "कि", "का", "को", "के", "ले", "मा", "हरु"]
suffix_re = re.compile("(" + "|".join(SUFFIXES) + ")$")

root_words = [w for w, c in vocab.items()
              if c >= MIN_WORD_FREQ and MIN_LEN <= len(w) <= MAX_LEN
              and not suffix_re.search(w)]
print(f"Candidate root-like words: {len(root_words)} / {len(vocab)} total vocabulary")

def levenshtein(a, b):
    n, m = len(a), len(b)
    if abs(n - m) > 1:
        return 99  # short-circuit: only interested in edit distance <= 1
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev = dp[0]; dp[0] = i
        for j in range(1, m + 1):
            tmp = dp[j]
            dp[j] = min(dp[j] + 1, dp[j - 1] + 1, prev + (a[i - 1] != b[j - 1]))
            prev = tmp
    return dp[m]

# Efficient candidate generation: two words that become identical after
# deleting one character from each are within edit distance 1 of each other
# (covers substitutions and single insertions/deletions). This avoids an
# infeasible all-pairs comparison over ~50k unique words.
variant_to_words = defaultdict(set)
for w in root_words:
    variant_to_words[w].add(w)
    for i in range(len(w)):
        variant_to_words[w[:i] + w[i + 1:]].add(w)

candidate_edges = set()
for variant, ws in variant_to_words.items():
    if len(ws) < 2:
        continue
    ws = list(ws)
    for i in range(len(ws)):
        for j in range(i + 1, len(ws)):
            if ws[i] != ws[j]:
                candidate_edges.add(tuple(sorted((ws[i], ws[j]))))

verified_edges = [(a, b) for a, b in candidate_edges if levenshtein(a, b) == 1]
print(f"Verified edit-distance-1 word pairs: {len(verified_edges)}")

parent = {}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]; x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb: parent[ra] = rb

for a, b in verified_edges:
    parent.setdefault(a, a); parent.setdefault(b, b)
    union(a, b)

components = defaultdict(set)
for w in parent:
    components[find(w)].add(w)

candidate_clusters = [c for c in components.values() if 2 <= len(c) <= 4]
candidate_clusters.sort(key=lambda c: -sum(vocab[w] for w in c))
selected_clusters = candidate_clusters[:NUM_CLUSTERS]

CLUSTERS = {}
for i, c in enumerate(selected_clusters, start=1):
    CLUSTERS[f"C{i:03d}"] = sorted(c, key=lambda w: -vocab[w])

clusters_df = pd.DataFrame([{"cluster_id": cid, "candidate_word": w, "corpus_frequency": vocab[w]}
                             for cid, ws in CLUSTERS.items() for w in ws])
print(f"\n{len(components)} raw components -> {len(candidate_clusters)} plausible (size 2-4) "
      f"-> using top {len(CLUSTERS)} clusters:\n")
print(clusters_df.to_string(index=False))


Candidate root-like words: 7792 / 49100 total vocabulary
Verified edit-distance-1 word pairs: 9854

610 raw components -> 561 plausible (size 2-4) -> using top 20 clusters:

cluster_id candidate_word  corpus_frequency
      C001         नेपाली              1177
      C001          नेपाल               716
      C002           त्यो               720
      C002           त्यस               380
      C002          त्यसै               109
      C002          त्यसो                21
      C003         अनुसार               624
      C003         अनुहार                14
      C004         जिल्ला               612
      C004         किल्ला                23
      C005        प्राप्त               528
      C005        प्रान्त                37
      C005       प्राप्ति                33
      C005       प्रशान्त                14
      C006          मुख्य               566
      C006         मुख्यत                 6
      C007        अवस्थित               511
      C007       अवस्थिति         

## Step 1 — Mine occurrences from the real corpus

Every `audio_index.tsv` row whose reference sentence contains a candidate
word (as a whole token) becomes one occurrence. Real Whisper ASR is attached
wherever `whisper_output.tsv` happens to cover the same `audio_id` (Input C,
"where available" — coverage will be small since that file wasn't built
around these specific clusters).


In [ ]:
word_to_cluster = {w: cid for cid, ws in CLUSTERS.items() for w in ws}
all_candidates_set = set(word_to_cluster)

t0 = time.time()
hits_by_word = defaultdict(list)
for aid, ref in zip(audio_index["audio_id"], audio_index["reference"].astype(str)):
    hit_words = set(ref.split()) & all_candidates_set
    for w in hit_words:
        hits_by_word[w].append((aid, ref))
print(f"Occurrence scan over {len(audio_index)} rows took {time.time()-t0:.2f}s")

occ_rows = []
for cluster_id, words in CLUSTERS.items():
    for w in words:
        pool = hits_by_word.get(w, [])
        if len(pool) > MAX_OCC_PER_CANDIDATE:
            idx = rng.choice(len(pool), size=MAX_OCC_PER_CANDIDATE, replace=False)
            pool = [pool[i] for i in idx]
        for aid, ref in pool:
            occ_rows.append({"occurrence_id": aid, "cluster_id": cluster_id,
                              "reference_word": w, "reference_transcript": ref})

occurrences_df = pd.DataFrame(occ_rows).drop_duplicates(
    subset=["occurrence_id", "cluster_id", "reference_word"])
print(f"Total occurrences mined: {len(occurrences_df)}")
print(occurrences_df["reference_word"].value_counts())


Occurrence scan over 157905 rows took 0.13s
Total occurrences mined: 4420
reference_word
नेपाली       150
नेपाल        150
त्यो         150
त्यस         150
अनुसार       150
मुख्य        150
प्राप्त      150
जिल्ला       150
सबैभन्दा     150
हुन्छन्      150
अवस्थित      150
समेत         150
राजनैतिक     150
दोस्रो       150
समिति        150
अधिकार       150
प्रसिद्ध     150
शिक्षा       150
पुरानो       150
विशेष        150
जन्म         150
दक्षिण       150
राजनीतिक     144
त्यसै        109
विशेषता       83
पुराना        81
राजनीति       65
अधिकारी       61
शिक्षण        59
दक्षिणी       49
सबभन्दा       41
शिक्षक        38
प्रान्त       37
प्राप्ति      33
हुनेछन्       30
मिति          28
अधिकतम        28
जन्मने        23
हुन्छन्।      23
किल्ला        23
दोश्रो        23
त्यसो         21
अनुहार        14
प्रशान्त      14
विशेषत        14
पुरानै        11
जन्मे         11
प्रसिद्धि      9
अवस्थिति       7
सयभन्दा        7
अधिकतर         7
मुख्यत         6
प्रशिद्ध       6
सचेत      

## Step 2 — Train/val/test split (occurrence-level; speaker-disjoint is not possible)

**Known deviation from the original spec:** none of the three provided files
contain a `speaker_id`, so the mandatory speaker-disjoint split cannot be
enforced here. This is a random 70/15/15 split over occurrences instead.
Swap this cell for a real speaker-based split the moment Person 1 supplies
speaker metadata — nothing downstream needs to change.


In [ ]:
n = len(occurrences_df)
perm = rng.permutation(n)
n_train, n_val = int(n * 0.70), int(n * 0.15)
split_labels = np.array(["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val))
occurrences_df = occurrences_df.iloc[perm].reset_index(drop=True)
occurrences_df["dataset_split"] = split_labels

print("NOTE: occurrence-level split, NOT speaker-disjoint (no speaker_id in the source data).")
print(occurrences_df["dataset_split"].value_counts())

asr_cols = ["audio_id", "hypothesis", "avg_logprob", "no_speech_prob"]
occurrences_df = occurrences_df.merge(
    whisper_output[asr_cols].rename(columns={"audio_id": "occurrence_id"}),
    on="occurrence_id", how="left",
)
print(f"\nOccurrences with real Whisper ASR results attached: "
      f"{occurrences_df['hypothesis'].notna().sum()} / {len(occurrences_df)}")


NOTE: occurrence-level split, NOT speaker-disjoint (no speaker_id in the source data).
dataset_split
train    3094
val       663
test      663
Name: count, dtype: int64

Occurrences with real Whisper ASR results attached: 9 / 4420


## PART A — Classical Acoustic (GMM) Evidence

**Known gap:** no `.flac` audio is reachable in this environment (the source
paths are a Windows filesystem, and no audio bytes were uploaded). The
extraction function below is fully real (`librosa`: MFCC + deltas, pYIN F0
treating unvoiced frames as missing, RMS, spectral centroid/bandwidth/
rolloff/contrast/flatness, ZCR) and will produce genuine acoustic evidence
the moment `.flac` files exist under `AUDIO_ROOT`, bucketed as
`AUDIO_ROOT/<first-2-chars-of-audio_id>/<audio_id>.flac`. Until then, this
section detects that no audio is available, says so plainly, and writes an
empty (but correctly-shaped) `acoustic_candidate_scores.csv` instead of
raising an error.


In [ ]:
def resolve_audio_path(audio_id):
    bucket = audio_id[:2]
    for candidate in (os.path.join(AUDIO_ROOT, bucket, f"{audio_id}.flac"),
                      os.path.join(AUDIO_ROOT, f"{audio_id}.flac")):
        if os.path.exists(candidate):
            return candidate
    return None

def extract_frame_features(audio_path, sr=16000, frame_length=0.025, hop_length=0.010, n_mfcc=13):
    """Real frame-level acoustic feature extraction, matching Person 1's declared schema."""
    y, _ = librosa.load(audio_path, sr=sr)
    if y.size == 0:
        return pd.DataFrame()
    n_fft = max(int(frame_length * sr), 32)
    hop = max(int(hop_length * sr), 16)

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, n_fft=n_fft, hop_length=hop)
    delta_mfcc = librosa.feature.delta(mfcc, order=1)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)
    f0, voiced_flag, _ = librosa.pyin(y, fmin=75.0, fmax=500.0, sr=sr,
                                       frame_length=max(2048, n_fft), hop_length=hop)
    f0 = np.where(voiced_flag, f0, np.nan)  # unvoiced -> missing, never an arbitrary pitch
    rms = librosa.feature.rms(y=y, frame_length=n_fft, hop_length=hop)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, n_fft=n_fft, hop_length=hop)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, n_fft=n_fft, hop_length=hop)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, n_fft=n_fft, hop_length=hop)[0]
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_fft=n_fft, hop_length=hop)
    flatness = librosa.feature.spectral_flatness(y=y, n_fft=n_fft, hop_length=hop)[0]
    zcr = librosa.feature.zero_crossing_rate(y=y, frame_length=n_fft, hop_length=hop)[0]

    n_frames = mfcc.shape[1]
    def fit(arr):
        arr = np.asarray(arr).reshape(-1)
        return arr[:n_frames] if len(arr) >= n_frames else np.pad(arr, (0, n_frames - len(arr)), constant_values=np.nan)

    data = {}
    for i in range(n_mfcc):
        data[f"mfcc_{i+1}"] = mfcc[i]
        data[f"delta_mfcc_{i+1}"] = delta_mfcc[i]
        data[f"delta2_mfcc_{i+1}"] = delta2_mfcc[i]
    data["f0"] = fit(f0)
    data["rms_energy"] = fit(rms)
    data["spectral_centroid"] = fit(centroid)
    data["spectral_bandwidth"] = fit(bandwidth)
    data["spectral_rolloff"] = fit(rolloff)
    for i in range(contrast.shape[0]):
        data[f"spectral_contrast_{i+1}"] = fit(contrast[i])
    data["spectral_flatness"] = fit(flatness)
    data["zcr"] = fit(zcr)
    out = pd.DataFrame(data)
    out.insert(0, "frame_idx", np.arange(len(out)))
    return out

occurrences_df["audio_path"] = occurrences_df["occurrence_id"].apply(resolve_audio_path)
occurrences_df["audio_available"] = occurrences_df["audio_path"].notna()
n_available = int(occurrences_df["audio_available"].sum())
print(f"Audio resolvable under AUDIO_ROOT='{AUDIO_ROOT}': {n_available} / {len(occurrences_df)} occurrences")

if n_available > 0:
    acoustic_frames = []
    for _, occ in occurrences_df[occurrences_df.audio_available].iterrows():
        feats = extract_frame_features(occ["audio_path"])
        feats.insert(0, "occurrence_id", occ["occurrence_id"])
        acoustic_frames.append(feats)
    acoustic_df = pd.concat(acoustic_frames, ignore_index=True) if acoustic_frames else pd.DataFrame()
else:
    acoustic_df = pd.DataFrame()
    print("\nNo audio files are reachable, so Part A cannot compute real acoustic evidence yet.")
    print(f"Set AUDIO_ROOT to the folder containing the .flac files "
          f"(bucketed as AUDIO_ROOT/<first-2-chars-of-id>/<audio_id>.flac) and re-run this cell.")
    print("acoustic_candidate_scores.csv will still be written, with the correct columns and")
    print("zero rows, so every later cell runs without error.")

print("acoustic_df shape:", acoustic_df.shape)


Audio resolvable under AUDIO_ROOT='/content/asr_nepali_audio': 0 / 4420 occurrences

No audio files are reachable, so Part A cannot compute real acoustic evidence yet.
Set AUDIO_ROOT to the folder containing the .flac files (bucketed as AUDIO_ROOT/<first-2-chars-of-id>/<audio_id>.flac) and re-run this cell.
acoustic_candidate_scores.csv will still be written, with the correct columns and
zero rows, so every later cell runs without error.
acoustic_df shape: (0, 0)


### Feature preprocessing + candidate GMMs

Structurally identical to the original spec (Sections 4–11): unusable/near-
zero-variance/highly-correlated feature removal → standardization →
per-candidate-word GMMs → hyperparameter selection on validation → softmax
normalization on test. When `acoustic_df` is empty (current situation), this
cell does nothing and leaves `acoustic_candidate_scores` empty with the
correct schema — it does not error.


In [ ]:
acoustic_candidate_scores = pd.DataFrame(
    columns=["occurrence_id", "cluster_id", "candidate_word", "GMM_loglikelihood", "GMM_normalized_score"]
)

if len(acoustic_df) > 0:
    frame_meta = acoustic_df.merge(
        occurrences_df[["occurrence_id", "cluster_id", "reference_word", "dataset_split"]],
        on="occurrence_id", how="left",
    )
    train_frames = frame_meta[frame_meta.dataset_split == "train"]
    feature_cols = [c for c in acoustic_df.columns if c not in ("occurrence_id", "frame_idx")]

    missing_frac = train_frames[feature_cols].isna().mean()
    usable = missing_frac[missing_frac <= 0.30].index.tolist()
    impute_values = train_frames[usable].mean()

    def impute(df):
        return df[usable].fillna(impute_values)

    vt = VarianceThreshold(threshold=1e-6)
    vt.fit(impute(train_frames))
    kept = [f for f, keep in zip(usable, vt.get_support()) if keep]

    corr = impute(train_frames)[kept].corr().abs()
    to_drop = set()
    for f1, f2 in combinations(kept, 2):
        if f1 in to_drop or f2 in to_drop:
            continue
        if corr.loc[f1, f2] > 0.95:
            to_drop.add(f2)
    final_features = [f for f in kept if f not in to_drop]
    print("Frozen feature set:", final_features)

    scaler = StandardScaler().fit(impute(train_frames)[final_features])
    def preprocess(df):
        return scaler.transform(impute(df)[final_features])

    ALL_CANDS = [w for ws in CLUSTERS.values() for w in ws]
    def frames_for(word, splits):
        ids = occurrences_df[(occurrences_df.reference_word == word) &
                              (occurrences_df.dataset_split.isin(splits))]["occurrence_id"]
        return frame_meta[frame_meta.occurrence_id.isin(ids)]

    best_k, best_cov, best_score = 1, "diag", -np.inf
    for k in [1, 2, 4]:
        for cov in ["diag", "full"]:
            lls = []
            for w in ALL_CANDS:
                tr, va = preprocess(frames_for(w, ["train"])), preprocess(frames_for(w, ["val"]))
                if len(tr) < 5 or len(va) < 1:
                    continue
                try:
                    gmm = GaussianMixture(n_components=min(k, max(1, len(tr)//5)),
                                           covariance_type=cov, random_state=0, reg_covar=1e-3).fit(tr)
                    lls.append(gmm.score(va))
                except Exception:
                    continue
            if lls and np.mean(lls) > best_score:
                best_score, best_k, best_cov = np.mean(lls), k, cov
    print(f"Selected GMM config: K={best_k}, covariance_type={best_cov}")

    final_gmms = {}
    for w in ALL_CANDS:
        dev = preprocess(frames_for(w, ["train", "val"]))
        if len(dev) < 5:
            continue
        final_gmms[w] = GaussianMixture(n_components=min(best_k, max(1, len(dev)//5)),
                                         covariance_type=best_cov, random_state=0, reg_covar=1e-3).fit(dev)

    rows = []
    for _, occ in occurrences_df[occurrences_df.dataset_split == "test"].iterrows():
        cands = [w for w in CLUSTERS[occ["cluster_id"]] if w in final_gmms]
        if len(cands) < 2:
            continue
        X = preprocess(frame_meta[frame_meta.occurrence_id == occ["occurrence_id"]])
        if len(X) == 0:
            continue
        ll = {w: final_gmms[w].score(X) for w in cands}
        norm = softmax(np.array(list(ll.values())))
        for w, s in zip(cands, norm):
            rows.append({"occurrence_id": occ["occurrence_id"], "cluster_id": occ["cluster_id"],
                         "candidate_word": w, "GMM_loglikelihood": ll[w], "GMM_normalized_score": s})
    acoustic_candidate_scores = pd.DataFrame(rows, columns=acoustic_candidate_scores.columns)

print("acoustic_candidate_scores:", acoustic_candidate_scores.shape)


acoustic_candidate_scores: (0, 5)


## PART B — Topic/Context (LDA) Evidence, on the real SLR54 corpus

Trained on a random subsample of `LDA_CORPUS_SIZE` real sentences from
`audio_index.tsv` (default 40,000 of the 157,905 available) — genuinely real
text throughout, unlike Part A.


In [ ]:
PUNCT_RE = re.compile(r"[।,.!?\"'()\-:;]")
def normalize_text(text):
    text = unicodedata.normalize("NFC", str(text))
    text = PUNCT_RE.sub(" ", text)
    return [t for t in text.split() if t]

corpus_idx = rng.choice(len(audio_index), size=min(LDA_CORPUS_SIZE, len(audio_index)), replace=False)
broad_sentences = audio_index["reference"].astype(str).iloc[corpus_idx].tolist()
broad_tokens = [normalize_text(s) for s in broad_sentences]

n_broad = len(broad_tokens)
broad_perm = rng.permutation(n_broad)
cut = int(n_broad * 0.85)
train_tok = [broad_tokens[i] for i in broad_perm[:cut]]
val_tok = [broad_tokens[i] for i in broad_perm[cut:]]

t0 = time.time()
dictionary = corpora.Dictionary(train_tok)
dictionary.filter_extremes(no_below=3, no_above=0.5)
train_bow = [dictionary.doc2bow(d) for d in train_tok]

K_GRID_LDA = [10, 20, 30]
lda_scores = {}
for k in K_GRID_LDA:
    lda_k = LdaModel(corpus=train_bow, id2word=dictionary, num_topics=k,
                      random_state=0, passes=5, alpha="auto")
    cm = CoherenceModel(model=lda_k, texts=train_tok, dictionary=dictionary, coherence="c_v")
    lda_scores[k] = cm.get_coherence()
print(f"LDA training/selection on {n_broad} real sentences took {time.time()-t0:.1f}s")
print("Coherence scores:", lda_scores)

BEST_K_LDA = max(lda_scores, key=lda_scores.get)
print("Selected (frozen) topic count:", BEST_K_LDA)

full_dev_tokens = train_tok + val_tok
dictionary = corpora.Dictionary(full_dev_tokens)
dictionary.filter_extremes(no_below=3, no_above=0.5)
dev_bow = [dictionary.doc2bow(d) for d in full_dev_tokens]
final_lda = LdaModel(corpus=dev_bow, id2word=dictionary, num_topics=BEST_K_LDA,
                      random_state=0, passes=10, alpha="auto")
topic_term_matrix = final_lda.get_topics()

all_cands = [w for ws in CLUSTERS.values() for w in ws]
in_vocab = [w for w in all_cands if w in dictionary.token2id]
print(f"LDA vocabulary size: {len(dictionary)} | candidate words present: {len(in_vocab)} / {len(all_cands)}")


LDA training/selection on 40000 real sentences took 71.7s
Coherence scores: {10: np.float64(0.6359587636093806), 20: np.float64(0.686453680798438), 30: np.float64(0.7165751975408726)}
Selected (frozen) topic count: 30
LDA vocabulary size: 6898 | candidate words present: 47 / 56


### Target masking, topic inference, candidate compatibility, OOV & normalization

Run on the TEST split of the real mined occurrences. The candidate word is
always removed from its own context before inference.


In [ ]:
def mask_target(sentence, target_word):
    return [t for t in normalize_text(sentence) if t != target_word]

test_occ = occurrences_df[occurrences_df.dataset_split == "test"]
topic_rows = []
for _, occ in test_occ.iterrows():
    cluster_id, cands = occ["cluster_id"], CLUSTERS[occ["cluster_id"]]
    context_tokens = mask_target(occ["reference_transcript"], occ["reference_word"])
    bow = dictionary.doc2bow(context_tokens)
    theta = final_lda.get_document_topics(bow, minimum_probability=0.0)
    theta_vec = np.array([p for _, p in sorted(theta, key=lambda x: x[0])])

    raw, oov = {}, {}
    for w in cands:
        if w in dictionary.token2id:
            p_w = topic_term_matrix[:, dictionary.token2id[w]]
            raw[w] = float(np.dot(theta_vec, p_w))
            oov[w] = False
        else:
            raw[w], oov[w] = 0.0, True

    total = sum(raw.values())
    norm = {w: s / total for w, s in raw.items()} if total > 0 else {w: 1.0 / len(cands) for w in cands}

    for w in cands:
        topic_rows.append({
            "occurrence_id": occ["occurrence_id"], "cluster_id": cluster_id, "candidate_word": w,
            "topic_distribution": theta_vec.tolist(),
            "topic_probability": raw[w], "topic_normalized_score": norm[w], "candidate_oov": oov[w],
        })

topic_candidate_scores = pd.DataFrame(topic_rows)
print("OOV rate among candidate evaluations:", topic_candidate_scores["candidate_oov"].mean())
topic_candidate_scores.drop(columns="topic_distribution").head(10)


OOV rate among candidate evaluations: 0.1365953109072375


,occurrence_id,cluster_id,candidate_word,topic_probability,topic_normalized_score,candidate_oov
0,1de72e419c,C009,हुन्छन्,0.000644,0.973335,False
1,1de72e419c,C009,हुनेछन्,0.000018,0.026665,False
2,1de72e419c,C009,हुन्छन्।,0.000000,0.000000,True
3,78ef1cfd9f,C013,दक्षिण,0.000845,0.983233,False
4,78ef1cfd9f,C013,दक्षिणी,0.000014,0.016767,False
5,7fb08ac5ec,C012,राजनैतिक,0.000407,0.604471,False
6,7fb08ac5ec,C012,राजनीतिक,0.000253,0.376890,False
7,7fb08ac5ec,C012,राजनीति,0.000013,0.018639,False
8,a0907cce26,C020,दोस्रो,0.000534,0.968307,False
9,a0907cce26,C020,दोश्रो,0.000017,0.031693,False


## PART C — Combined Candidate-Level Evidence


In [ ]:
traceability = occurrences_df[["occurrence_id", "reference_word", "dataset_split"]]

candidate_evidence = (
    topic_candidate_scores[["occurrence_id", "cluster_id", "candidate_word", "topic_normalized_score"]]
    .merge(acoustic_candidate_scores[["occurrence_id", "candidate_word", "GMM_normalized_score"]],
           on=["occurrence_id", "candidate_word"], how="left")
    .rename(columns={"GMM_normalized_score": "GMM_score", "topic_normalized_score": "topic_score"})
    .merge(traceability, on="occurrence_id", how="left")
)
candidate_evidence = candidate_evidence[["occurrence_id", "cluster_id", "candidate_word",
                                          "GMM_score", "topic_score", "reference_word", "dataset_split"]]
print("Rows with GMM_score available:", candidate_evidence["GMM_score"].notna().sum(),
      "/", len(candidate_evidence))
candidate_evidence.head(10)


Rows with GMM_score available: 0 / 2014


,occurrence_id,cluster_id,candidate_word,GMM_score,topic_score,reference_word,dataset_split
0,1de72e419c,C009,हुन्छन्,NaN,0.973335,हुन्छन्,test
1,1de72e419c,C009,हुनेछन्,NaN,0.026665,हुन्छन्,test
2,1de72e419c,C009,हुन्छन्।,NaN,0.000000,हुन्छन्,test
3,78ef1cfd9f,C013,दक्षिण,NaN,0.983233,दक्षिण,test
4,78ef1cfd9f,C013,दक्षिणी,NaN,0.016767,दक्षिण,test
5,7fb08ac5ec,C012,राजनैतिक,NaN,0.604471,राजनीति,test
6,7fb08ac5ec,C012,राजनीतिक,NaN,0.376890,राजनीति,test
7,7fb08ac5ec,C012,राजनीति,NaN,0.018639,राजनीति,test
8,a0907cce26,C020,दोस्रो,NaN,0.968307,दोस्रो,test
9,a0907cce26,C020,दोश्रो,NaN,0.031693,दोस्रो,test


### Evidence-ablation diagnostics

Diagnostic only, never a final prediction. GMM-only is skipped (reported as
`None`, not an error) whenever no acoustic evidence exists.


In [ ]:
def top1_match_rate(df, score_col):
    d = df.dropna(subset=[score_col])
    if d.empty:
        return None
    picks = d.loc[d.groupby("occurrence_id")[score_col].idxmax()]
    return (picks["candidate_word"] == picks["reference_word"]).mean()

lda_rate = top1_match_rate(candidate_evidence, "topic_score")
gmm_rate = top1_match_rate(candidate_evidence, "GMM_score")
print(f"LDA-only top-1 diagnostic match rate: {lda_rate}")
print(f"GMM-only top-1 diagnostic match rate: {gmm_rate}"
      + ("  (skipped -- no acoustic evidence available yet)" if gmm_rate is None else ""))


LDA-only top-1 diagnostic match rate: 0.6525679758308157
GMM-only top-1 diagnostic match rate: None  (skipped -- no acoustic evidence available yet)


## Deliverables + artifacts


In [ ]:
clusters_df.to_csv(os.path.join(OUTPUT_DIR, "confusable_word_clusters.csv"), index=False)
occurrences_df.drop(columns=["audio_path"], errors="ignore").to_csv(
    os.path.join(OUTPUT_DIR, "openslr_occurrences.csv"), index=False)
acoustic_candidate_scores.to_csv(os.path.join(OUTPUT_DIR, "acoustic_candidate_scores.csv"), index=False)

topic_export = topic_candidate_scores.copy()
topic_export["topic_distribution"] = topic_export["topic_distribution"].apply(json.dumps)
topic_export.to_csv(os.path.join(OUTPUT_DIR, "topic_candidate_scores.csv"), index=False)

candidate_evidence.to_csv(os.path.join(OUTPUT_DIR, "candidate_evidence.csv"), index=False)

final_lda.save(os.path.join(ARTIFACT_DIR, "lda_model.gensim"))
dictionary.save(os.path.join(ARTIFACT_DIR, "lda_dictionary.gensim"))
with open(os.path.join(ARTIFACT_DIR, "lda_config.json"), "w", encoding="utf-8") as f:
    json.dump({"selected_num_topics": int(BEST_K_LDA),
               "coherence_scores": {int(k): v for k, v in lda_scores.items()}},
              f, ensure_ascii=False, indent=2)

if len(acoustic_df) > 0:
    joblib.dump({"final_features": final_features, "scaler": scaler,
                 "selected_K": best_k, "selected_covariance_type": best_cov},
                os.path.join(ARTIFACT_DIR, "gmm_preprocessing_config.joblib"))
    for w, gmm in final_gmms.items():
        joblib.dump(gmm, os.path.join(ARTIFACT_DIR, f"gmm_{w}.joblib"))

print("Saved deliverables to:", OUTPUT_DIR)
for root, _, files_ in os.walk(OUTPUT_DIR):
    for fn in sorted(files_):
        print(" -", os.path.join(root, fn))


Saved deliverables to: /content/person2_outputs
 - /content/person2_outputs/acoustic_candidate_scores.csv
 - /content/person2_outputs/candidate_evidence.csv
 - /content/person2_outputs/confusable_word_clusters.csv
 - /content/person2_outputs/openslr_occurrences.csv
 - /content/person2_outputs/topic_candidate_scores.csv
 - /content/person2_outputs/artifacts/lda_config.json
 - /content/person2_outputs/artifacts/lda_dictionary.gensim
 - /content/person2_outputs/artifacts/lda_model.gensim
 - /content/person2_outputs/artifacts/lda_model.gensim.expElogbeta.npy
 - /content/person2_outputs/artifacts/lda_model.gensim.id2word
 - /content/person2_outputs/artifacts/lda_model.gensim.state


## Download the outputs (optional, Colab only)


In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/person2_outputs", "zip", OUTPUT_DIR)
files.download(zip_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

- **Real data used throughout Part B** (LDA trained + evaluated on actual
  SLR54 sentences from `audio_index.tsv`).
- **Part A is real but currently empty**: no `.flac` audio was reachable in
  this run, so `GMM_score` is `NaN` for every row. Set `AUDIO_ROOT` once real
  audio is available and re-run — the same code will then produce real
  acoustic evidence with no other changes needed.
- **Confusable clusters were bootstrapped from real vocabulary**, not
  hand-picked, since no curated list was supplied. Swap in a linguist-curated
  `confusable_word_clusters.csv` at any time — everything downstream reads
  that table the same way regardless of its origin.
- **The split is occurrence-level, not speaker-disjoint**, because none of
  the three files carry `speaker_id`. Get that from Person 1 and swap the
  split cell in for a proper one.
- `candidate_evidence.csv` is ready for Person 3 as-is; Person 2 does not
  select a final candidate anywhere in this notebook.
